In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

INPUT_DIR = r"D:\cells\А549\48"
OUTPUT_DIR = r"D:\cells\A549_png\48"
CONVERT_TO_GRAYSCALE = True
NORMALIZE_16BIT = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

def normalize_to_8bit(img_array):

    img_array = img_array.astype(np.float32)

    min_val = np.min(img_array)
    max_val = np.max(img_array)

    if max_val - min_val == 0:
        return np.zeros_like(img_array, dtype=np.uint8)

    img_array = (img_array - min_val) / (max_val - min_val)
    img_array = (img_array * 255).clip(0, 255)

    return img_array.astype(np.uint8)

tiff_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith((".tif", ".tiff"))]

print(f"Найдено файлов: {len(tiff_files)}")

for filename in tqdm(tiff_files):
    input_path = os.path.join(INPUT_DIR, filename)
    output_name = os.path.splitext(filename)[0] + ".png"
    output_path = os.path.join(OUTPUT_DIR, output_name)

    with Image.open(input_path) as img:

        img_array = np.array(img)

        if CONVERT_TO_GRAYSCALE and len(img_array.shape) == 3:
            img_array = np.mean(img_array, axis=2)

        if NORMALIZE_16BIT and img_array.dtype != np.uint8:
            img_array = normalize_to_8bit(img_array)

        img_out = Image.fromarray(img_array)
        img_out.save(output_path, format="PNG")

print("Готово.")

Найдено файлов: 226


100%|██████████| 226/226 [15:42<00:00,  4.17s/it]

Готово.


In [ ]:
from pathlib import Path
import random
import shutil

SRC_ROOT = Path(r"ML\datasets")   
DST_ROOT = Path(r"dataset")  
CLASSES = ["24", "48", "72", "healthy"]

SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15

MIN_VAL_PER_CLASS = 10
MIN_TEST_PER_CLASS = 10


random.seed(SEED)

def list_pngs(class_dir: Path):
    return sorted([p for p in class_dir.rglob("*.png") if p.is_file()])

def ensure_dirs():
    for split in ["train", "val", "test"]:
        for c in CLASSES:
            (DST_ROOT / split / c).mkdir(parents=True, exist_ok=True)

def split_counts(n: int):
    n_val = max(int(round(n * VAL_FRAC)), MIN_VAL_PER_CLASS)
    n_test = max(int(round(n * TEST_FRAC)), MIN_TEST_PER_CLASS)

    if n_val + n_test >= n:
        n_val = min(n_val, n - 2)
        n_test = min(n_test, n - 1 - n_val)

    n_train = n - n_val - n_test
    return n_train, n_val, n_test

def copy_files(files, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    for src in files:
        dst = out_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)

missing = [c for c in CLASSES if not (SRC_ROOT / c).is_dir()]
if missing:
    raise SystemExit(f"Не найдены папки классов в {SRC_ROOT}: {missing}")

ensure_dirs()

summary = {}

for c in CLASSES:
    files = list_pngs(SRC_ROOT / c)
    n = len(files)
    if n == 0:
        raise SystemExit(f"Класс {c} пустой")

    random.shuffle(files)
    n_train, n_val, n_test = split_counts(n)

    val_files = files[:n_val]
    test_files = files[n_val:n_val + n_test]
    train_files = files[n_val + n_test:]

    copy_files(train_files, DST_ROOT / "train" / c)
    copy_files(val_files,   DST_ROOT / "val"   / c)
    copy_files(test_files,  DST_ROOT / "test"  / c)

    summary[c] = {"train": len(train_files), "val": len(val_files), "test": len(test_files), "total": n}

print(f"SRC_ROOT = {SRC_ROOT.resolve()}")
print(f"DST_ROOT = {DST_ROOT.resolve()}\n")

print("Распределение по классам:")
tot_train = tot_val = tot_test = tot_all = 0
for c in CLASSES:
    s = summary[c]
    print(f"  {c:>7}: train={s['train']:>3}  val={s['val']:>3}  test={s['test']:>3}  total={s['total']}")
    tot_train += s["train"]
    tot_val   += s["val"]
    tot_test  += s["test"]
    tot_all   += s["total"]

print("\nИТОГО:")
print(f"  train={tot_train}  val={tot_val}  test={tot_test}  total={tot_all}")

SRC_ROOT = D:\cells\A549_png
DST_ROOT = D:\cells\dataset

Распределение по классам:
       24: train=154  val= 33  test= 33  total=220
       48: train=158  val= 34  test= 34  total=226
       72: train= 52  val= 11  test= 11  total=74
  healthy: train= 41  val= 10  test= 10  total=61

ИТОГО:
  train=405  val=88  test=88  total=581


In [9]:
from pathlib import Path
from collections import Counter
import numpy as np
import tensorflow as tf

DATA_ROOT = Path(r"D:\cells\ML\datasets\dataset")
CLASS_NAMES = ["24", "48", "72", "healthy"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

MODEL_SIZE = 512          # размер входа модели
RAW_PATCH_SIZE = 1024     # patch из исходного изображения
BATCH_SIZE = 8           # на CPU лучше 2-4
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

tf.random.set_seed(SEED)

def collect_split(split_name):
    split_root = DATA_ROOT / split_name
    paths, labels = [], []

    for cls_name in CLASS_NAMES:
        class_dir = split_root / cls_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Не найдена папка: {class_dir}")

        for ext in ("*.png", "*.jpg", "*.jpeg"):
            for p in sorted(class_dir.glob(ext)):
                paths.append(str(p))
                labels.append(CLASS_TO_IDX[cls_name])

    return paths, labels


train_paths, train_labels = collect_split("train")
val_paths, val_labels = collect_split("val")
test_paths, test_labels = collect_split("test")

print("train:", len(train_paths))
print("val:  ", len(val_paths))
print("test: ", len(test_paths))

train: 405
val:   88
test:  88


In [10]:
def read_image(path):
    # Считываем как grayscale, потом дублируем в 3 канала
    img = tf.io.read_file(path)
    img = tf.io.decode_png(img, channels=1)  # [H, W, 1]
    img = tf.image.grayscale_to_rgb(img)     # [H, W, 3]
    img = tf.cast(img, tf.float32)           # оставляем диапазон 0..255
    return img


def pad_to_min_size(img, min_size):
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]

    pad_h = tf.maximum(0, min_size - h)
    pad_w = tf.maximum(0, min_size - w)

    img = tf.pad(
        img,
        paddings=[
            [pad_h // 2, pad_h - pad_h // 2],
            [pad_w // 2, pad_w - pad_w // 2],
            [0, 0]
        ],
        mode="REFLECT"
    )
    return img


def random_patch_from_fullres(img):
    img = pad_to_min_size(img, RAW_PATCH_SIZE)
    img = tf.image.random_crop(img, size=[RAW_PATCH_SIZE, RAW_PATCH_SIZE, 3])
    img = tf.image.resize(img, [MODEL_SIZE, MODEL_SIZE], method="bilinear")
    return img


def center_patch_from_fullres(img):
    img = pad_to_min_size(img, RAW_PATCH_SIZE)

    h = tf.shape(img)[0]
    w = tf.shape(img)[1]

    offset_h = (h - RAW_PATCH_SIZE) // 2
    offset_w = (w - RAW_PATCH_SIZE) // 2

    img = tf.image.crop_to_bounding_box(
        img,
        offset_h,
        offset_w,
        RAW_PATCH_SIZE,
        RAW_PATCH_SIZE
    )
    img = tf.image.resize(img, [MODEL_SIZE, MODEL_SIZE], method="bilinear")
    return img


def preprocess_train(path, label):
    img = read_image(path)
    img = random_patch_from_fullres(img)
    return img, label


def preprocess_eval(path, label):
    img = read_image(path)
    img = center_patch_from_fullres(img)
    return img, label

In [11]:
def make_dataset(paths, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(preprocess_train, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(preprocess_eval, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)

In [12]:
counts = Counter(train_labels)
num_classes = len(CLASS_NAMES)
total = len(train_labels)

class_weight = {
    cls_idx: total / (num_classes * count)
    for cls_idx, count in counts.items()
}

print("train class counts:", counts)
print("class_weight:", class_weight)

train class counts: Counter({1: 158, 0: 154, 2: 52, 3: 41})
class_weight: {0: 0.6574675324675324, 1: 0.6408227848101266, 2: 1.9471153846153846, 3: 2.4695121951219514}


In [13]:
class SparseMacroF1(tf.keras.metrics.Metric):
    def __init__(self, num_classes, name="macro_f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.cm = self.add_weight(
            name="conf_matrix",
            shape=(num_classes, num_classes),
            initializer="zeros",
            dtype=tf.float32
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.reshape(tf.cast(y_true, tf.int32), [-1])
        y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)

        cm_batch = tf.math.confusion_matrix(
            y_true,
            y_pred,
            num_classes=self.num_classes,
            dtype=tf.float32
        )
        self.cm.assign_add(cm_batch)

    def result(self):
        tp = tf.linalg.diag_part(self.cm)
        fp = tf.reduce_sum(self.cm, axis=0) - tp
        fn = tf.reduce_sum(self.cm, axis=1) - tp

        precision = tf.math.divide_no_nan(tp, tp + fp)
        recall = tf.math.divide_no_nan(tp, tp + fn)
        f1 = tf.math.divide_no_nan(2.0 * precision * recall, precision + recall)

        return tf.reduce_mean(f1)

    def reset_state(self):
        self.cm.assign(tf.zeros_like(self.cm))

In [14]:
from tensorflow.keras import layers, Model

def build_model():
    inputs = tf.keras.Input(shape=(MODEL_SIZE, MODEL_SIZE, 3))

    x = layers.RandomFlip("horizontal_and_vertical")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomContrast(0.10)(x)

    x = layers.Rescaling(scale=1.0 / 127.5, offset=-1.0)(x)

    base = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        include_preprocessing=False
    )
    base.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

    model = Model(inputs, outputs)
    return model, base


model, base = build_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
        SparseMacroF1(num_classes=len(CLASS_NAMES), name="macro_f1"),
    ]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 512, 512, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_1               │ (None, 512, 512, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 16, 16, 1280)   │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,336,484 (77.58 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 20,331,360 (77.56 MB)

In [15]:
callbacks_stage1 = [
    tf.keras.callbacks.ModelCheckpoint(
        "a549_patch_stage1.keras",
        monitor="val_macro_f1",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_macro_f1",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=callbacks_stage1
)

Epoch 1/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.4479 - loss: 1.1963 - macro_f1: 0.3742 - top2_acc: 0.7833
Epoch 1: val_macro_f1 improved from -inf to 0.81173, saving model to a549_patch_stage1.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - acc: 0.4505 - loss: 1.1950 - macro_f1: 0.3771 - top2_acc: 0.7842 - val_acc: 0.8295 - val_loss: 0.8512 - val_macro_f1: 0.8117 - val_top2_acc: 0.8864 - learning_rate: 0.0010
Epoch 2/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.7033 - loss: 0.8747 - macro_f1: 0.6603 - top2_acc: 0.8984
Epoch 2: val_macro_f1 did not improve from 0.81173
51/51 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - acc: 0.7040 - loss: 0.8729 - macro_f1: 0.6607 - top2_acc: 0.8986 - val_acc: 0.8295 - val_loss: 0.7309 - val_macro_f1: 0.8068 - val_top2_acc: 0.8864 - learning_rate: 0.0010
Epoch 3/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.7756 - loss: 0.6837 - macro_f1: 0.7204 - top2_acc: 0.9390
Epoch 3: val_macro_f1 improved from 0.81173 to 0.81318, saving model to a549_pat

In [16]:
base.trainable = True

for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

freeze_until = int(0.65 * len(base.layers))
for layer in base.layers[:freeze_until]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
        SparseMacroF1(num_classes=len(CLASS_NAMES), name="macro_f1"),
    ]
)

callbacks_stage2 = [
    tf.keras.callbacks.ModelCheckpoint(
        "a549_patch_finetuned.keras",
        monitor="val_macro_f1",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_macro_f1",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight,
    callbacks=callbacks_stage2
)

Epoch 1/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.8658 - loss: 0.3909 - macro_f1: 0.8361 - top2_acc: 0.9510
Epoch 1: val_macro_f1 improved from -inf to 0.81509, saving model to a549_patch_finetuned.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 162s 3s/step - acc: 0.8657 - loss: 0.3920 - macro_f1: 0.8359 - top2_acc: 0.9511 - val_acc: 0.8295 - val_loss: 0.6479 - val_macro_f1: 0.8151 - val_top2_acc: 0.9205 - learning_rate: 1.0000e-05
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.8530 - loss: 0.4199 - macro_f1: 0.8128 - top2_acc: 0.9239
Epoch 2: val_macro_f1 improved from 0.81509 to 0.83239, saving model to a549_patch_finetuned.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 138s 3s/step - acc: 0.8536 - loss: 0.4187 - macro_f1: 0.8136 - top2_acc: 0.9245 - val_acc: 0.8523 - val_loss: 0.5332 - val_macro_f1: 0.8324 - val_top2_acc: 0.9205 - learning_rate: 1.0000e-05
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.8855 - loss: 0.3124 - macro_f1: 0.8491 - top2_acc: 0.9681
Epoch 3: val_macro_f1

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.models import load_model

model = tf.keras.models.load_model(
    "D:\cells\service\models\RSV_A549_cls.keras",
    compile=False
)


<>:6: SyntaxWarning: invalid escape sequence '\c'
<>:6: SyntaxWarning: invalid escape sequence '\c'
C:\Users\Glush\AppData\Local\Temp\ipykernel_16032\1317516581.py:6: SyntaxWarning: invalid escape sequence '\c'
  "D:\cells\service\models\RSV_A549_cls.keras",
C:\Users\Glush\AppData\Local\Temp\ipykernel_16032\1317516581.py:6: SyntaxWarning: invalid escape sequence '\c'
  "D:\cells\service\models\RSV_A549_cls.keras",


TypeError: Could not locate class 'SparseMacroF1'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': None, 'class_name': 'SparseMacroF1', 'config': {'name': 'macro_f1', 'dtype': 'float32'}, 'registered_name': 'SparseMacroF1'}

In [22]:
test_metrics = model.evaluate(test_ds, return_dict=True)
print(test_metrics)

ValueError: You must call `compile()` before using the model.

In [3]:
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
def read_image(path):
    # Считываем как grayscale, потом дублируем в 3 канала
    img = tf.io.read_file(path)
    img = tf.io.decode_png(img, channels=1)  # [H, W, 1]
    img = tf.image.grayscale_to_rgb(img)     # [H, W, 3]
    img = tf.cast(img, tf.float32)           # оставляем диапазон 0..255
    return img

def pad_to_min_size(img, min_size):
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]

    pad_h = tf.maximum(0, min_size - h)
    pad_w = tf.maximum(0, min_size - w)

    img = tf.pad(
        img,
        paddings=[
            [pad_h // 2, pad_h - pad_h // 2],
            [pad_w // 2, pad_w - pad_w // 2],
            [0, 0]
        ],
        mode="REFLECT"
    )
    return img
def five_crops_from_fullres(img):
    RAW_PATCH_SIZE = 1024
    img = pad_to_min_size(img, RAW_PATCH_SIZE)

    h = tf.shape(img)[0]
    w = tf.shape(img)[1]

    y0 = 0
    x0 = 0
    y1 = h - RAW_PATCH_SIZE
    x1 = w - RAW_PATCH_SIZE
    yc = (h - RAW_PATCH_SIZE) // 2
    xc = (w - RAW_PATCH_SIZE) // 2

    crops = [
        tf.image.crop_to_bounding_box(img, y0, x0, RAW_PATCH_SIZE, RAW_PATCH_SIZE),  # top-left
        tf.image.crop_to_bounding_box(img, y0, x1, RAW_PATCH_SIZE, RAW_PATCH_SIZE),  # top-right
        tf.image.crop_to_bounding_box(img, y1, x0, RAW_PATCH_SIZE, RAW_PATCH_SIZE),  # bottom-left
        tf.image.crop_to_bounding_box(img, y1, x1, RAW_PATCH_SIZE, RAW_PATCH_SIZE),  # bottom-right
        tf.image.crop_to_bounding_box(img, yc, xc, RAW_PATCH_SIZE, RAW_PATCH_SIZE),  # center
    ]

    crops = [tf.image.resize(c, [MODEL_SIZE, MODEL_SIZE], method="bilinear") for c in crops]
    crops = tf.stack(crops, axis=0)  # [5, 512, 512, 3]
    return crops


def predict_multicrop(image_path):
    img = read_image(image_path)
    crops = five_crops_from_fullres(img)

    probs = model.predict(crops, verbose=0)   # [5, num_classes]
    mean_probs = probs.mean(axis=0)

    pred_idx = int(np.argmax(mean_probs))
    return pred_idx, mean_probs

In [4]:
y_true = []
y_pred = []

for path, true_label in zip(test_paths, test_labels):
    pred_idx, probs = predict_multicrop(path)
    y_true.append(true_label)
    y_pred.append(pred_idx)

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

NameError: name 'test_paths' is not defined

In [5]:
predict_multicrop("D:\cells\ML\datasets\dataset\\test\healthy\TUC-20251223094955514.png")

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\Glush\AppData\Local\Temp\ipykernel_16032\260607599.py:1: SyntaxWarning: invalid escape sequence '\c'
  predict_multicrop("D:\cells\ML\datasets\dataset\\test\healthy\TUC-20251223094955514.png")
C:\Users\Glush\AppData\Local\Temp\ipykernel_16032\260607599.py:1: SyntaxWarning: invalid escape sequence '\c'
  predict_multicrop("D:\cells\ML\datasets\dataset\\test\healthy\TUC-20251223094955514.png")


NameError: name 'MODEL_SIZE' is not defined